In [0]:
source_catalog = "healthverity_claims_sample_patient_dataset.hv_claims_sample"
index = "john_snow_labs_icd_9_icd_10_and_clinical_classification_codes.icd_9_icd_10_and_clinical_classification_codes"

ENROLLMENT = f"{source_catalog}.enrollment"
CLAIMS      = f"{source_catalog}.medical_claim"
PROCEDURE   = f"{source_catalog}.procedure"
DIAGNOSIS   = f"{source_catalog}.diagnosis"
PROVIDER = f"{source_catalog}.provider"
RX = f"{source_catalog}.pharmacy_claim"
DX_DIM = f"{index}.clinical_classification_software_for_icd_10_cm"

stage0 = "workspace.default.stage0"
stage1 = "workspace.default.stage1"
stage2 = "workspace.default.stage2"
stage3 = "workspace.default.stage3"

In [0]:
from pyspark.sql import functions as F

# Load + alias
e = spark.table(ENROLLMENT).alias("e")
c = spark.table(CLAIMS).alias("c")

# Helper date columns (used for comparisons only)
e2 = (
    e.withColumn("date_start_d", F.to_date(F.col("e.date_start")))
     .withColumn("date_end_d",   F.to_date(F.col("e.date_end")))
     .alias("e")
)

c2 = (
    c.withColumn("date_service_d", F.to_date(F.col("c.date_service")))
     .alias("c")
)

# Build output column lists:
e_cols_out = [col for col in e.columns if col not in {"date_start_d", "date_end_d"}]  # these helper cols aren't in e.columns anyway
# Note: date_start_d/date_end_d are added cols, so we exclude them later explicitly.

# Claim columns to keep in joined outputs = claims columns not already on enrollment
overlap_cols = set(e.columns) & set(c.columns)
c_cols_keep_joined = [col for col in c.columns if col not in overlap_cols]

# Final select lists (qualified) for joined outputs
select_joined = (
    [F.col(f"e.{col}") for col in e.columns] +
    [F.col(f"c.{col}") for col in c_cols_keep_joined]
)

# --- Matched ---
matched = (
    e2.join(
        c2,
        (F.col("e.patient_id") == F.col("c.patient_id")) &
        (F.col("c.date_service_d") >= F.col("e.date_start_d")) &
        (F.col("c.date_service_d") <= F.col("e.date_end_d")),
        how="inner"
    )
    .select(*select_joined)
    .withColumn("member_match", F.lit("matched"))
)

# --- Ineligible ---
ineligible = (
    e2.join(
        c2,
        (F.col("e.patient_id") == F.col("c.patient_id")) &
        (
            (F.col("c.date_service_d") < F.col("e.date_start_d")) |
            (F.col("c.date_service_d") > F.col("e.date_end_d"))
        ),
        how="inner"
    )
    .select(*select_joined)
    .withColumn("member_match", F.lit("ineligible"))
)

# --- Unmatched (claims patient_id not in enrollment) ---
unmatched = (
    c2.join(
        e2,
        F.col("c.patient_id") == F.col("e.patient_id"),
        how="left_anti"
    )
    .drop("date_service_d")              # drop helper col
    .withColumn("member_match", F.lit("unmatched"))
)

matched = matched.drop("date_start_d", "date_end_d", "date_service_d")
ineligible = ineligible.drop("date_start_d", "date_end_d", "date_service_d")

first_df = (
    matched
    .unionByName(ineligible, allowMissingColumns=True)
    .unionByName(unmatched, allowMissingColumns=True)
)

(
    first_df
    .write
    .format("delta")
    .option("mergeSchema", "true")
    .mode("overwrite")
    .saveAsTable(stage0)
)
    
display(first_df.limit(20))

In [0]:
from pyspark.sql import functions as F

def left_join_enrich(
    base_df,
    right_df,
    join_key="claim_id",
    base_alias="s",
    right_alias="r",
    prefer_right=True,
    how="left"
):
    s = base_df.alias(base_alias)
    r = right_df.alias(right_alias)

    join_cond = F.col(f"{base_alias}.{join_key}") == F.col(f"{right_alias}.{join_key}")

    overlap = (set(base_df.columns) & set(right_df.columns)) - {join_key}
    right_only = [c for c in right_df.columns if c not in overlap and c != join_key]

    joined = s.join(r, on=join_cond, how=how)

    # Keep base columns, but for overlapping ones coalesce
    base_cols = []
    for c in base_df.columns:
        if c in overlap:
            if prefer_right:
                base_cols.append(
                    F.coalesce(F.col(f"{base_alias}.{c}"), F.col(f"{right_alias}.{c}")).alias(c)
                )
            else:
                base_cols.append(
                    F.coalesce(F.col(f"{right_alias}.{c}"), F.col(f"{base_alias}.{c}")).alias(c)
                )
        else:
            base_cols.append(F.col(f"{base_alias}.{c}").alias(c))

    # Append right-only columns
    right_cols = [F.col(f"{right_alias}.{c}").alias(c) for c in right_only]

    return joined.select(*(base_cols + right_cols))


# ---- usage ----
joins = [
    ("p",   spark.table(PROCEDURE)),
    ("dx",  spark.table(DIAGNOSIS)),
    ("prv", spark.table(PROVIDER)),
]

df = spark.table("workspace.default.stage0")

for alias, right in joins:
    df = left_join_enrich(df, right, join_key="claim_id", base_alias="s", right_alias=alias, prefer_right=True)

# Write out
(
    df.write
      .format("delta")
      .option("mergeSchema", "true")
      .mode("overwrite")
      .saveAsTable(stage1)
)

display(df.limit(10))


In [0]:
from pyspark.sql import functions as F

df = spark.table(stage1)
dim = spark.table(DX_DIM)

enriched = df.join(
    dim.select("ICD10CM_Code", "ICD10CM_Code_Description"),
    df["diagnosis_code"] == dim["ICD10CM_Code"]
)
enriched = enriched.drop("ICD10CM_Code")

# Write out
(
    enriched.write
      .format("delta")
      .option("mergeSchema", "true")
      .mode("overwrite")
      .saveAsTable(stage2)
)

display(enriched.limit(10))


In [0]:
from pyspark.sql import functions as F

scrub = spark.table(stage2)

# Pivot for procedure_qual
proc_pivot = (
    scrub.groupBy("claim_id")
    .pivot("procedure_qual")
    .agg(F.first("procedure_code"))
)

# Pivot for npi_role
npi_pivot = (
    scrub.groupBy("claim_id")
    .pivot("npi_role")
    .agg(
        F.first("npi").alias("npi"),
        F.first("taxonomy_code").alias("taxonomy_code")
    )
)

# Rename columns with invalid characters
for col in npi_pivot.columns:
    if "(" in col or ")" in col or "," in col or " " in col:
        new_col = col.replace("(", "_").replace(")", "").replace(",", "_").replace(" ", "_")
        npi_pivot = npi_pivot.withColumnRenamed(col, new_col)

scrub = (
    scrub.withColumn("line_allowed", F.round("line_allowed", 2))
         .withColumn("line_charge", F.round("line_charge", 2))
)

cols_to_drop = ["procedure_code", "procedure_qual", "npi", "taxonomy_code"]
scrub = (
    scrub.drop(*cols_to_drop)
    .join(proc_pivot, on="claim_id", how="left")
    .join(npi_pivot, on="claim_id", how="left")
)

scrub = (
    scrub.dropDuplicates(["claim_id", "patient_id"])
    .orderBy("claim_id", "patient_id")
)
(
    scrub.write
      .format("delta")
      .option("mergeSchema", "true")
      .mode("overwrite")
      .saveAsTable(stage3)
)

OUTPUT_TABLE = "workspace.default.stage3"

# write your table
scrub.write.mode("overwrite").saveAsTable(OUTPUT_TABLE)

# pass string to Task 2
dbutils.jobs.taskValues.set(
    key="output_table",
    value=OUTPUT_TABLE
)

display(scrub.limit(10))